# Knowledge distillation: E0–E3

A frozen, pretrained **teacher** (ResNet-50 on ImageNet-1k) supervises two **students**
trained from scratch: a CNN (**ResNet-18**) and a ViT (**ViT-Tiny**). The first trial is
on Tiny ImageNet: 10 named classes plus one "other" class. The same notebook runs on
full ImageNet-1k by changing two lines (see *Setup*). Each experiment changes **one**
thing:

| Section | Question | Trains a model? |
|---|---|---|
| E0a | How good is each teacher on this data, and how should its "other" logit be corrected? | No: inference only |
| E0b | How good is each student **without** a teacher? (the baselines) | Yes: CE only |
| E0c | Does a head trained on the frozen teacher beat its corrected ImageNet logits? Which teacher variant gives the best student? | Heads only (seconds), then one short KD run per variant |
| E1 | Does logit KD help, and with which temperature T and weight α? | Yes |
| E2 | Is another logit loss (DKD, DIST) better than plain KD? | Yes |
| E3 | Does matching the teacher's **embedding** help, even with no labels? | Yes |

The reusable parts (datasets, models, losses, training loop) are in
[`parts.py`](parts.py), built only from `lwfm` and torch. This notebook only chooses
which parts to combine. The reasoning behind each choice is in
[`kd_plan_E0-E3.md`](../../kd_plan_E0-E3.md).

**Runs.** Every run writes its own folder under `runs/<dataset>/<split>/`:

- `teachers/<teacher>/<variant>/`: a teacher scored alone (E0a, E0c)
- `<teacher variant>/<experiment>/<student>/<stage>/<name>_s<seed>/`: a training run.
  The variant is `no-teacher` for the baselines, and e.g. `resnet50-linear` for KD, so
  runs from different teachers are never mixed up or reused by mistake.

Each folder holds the
settings (`manifest.json`), every logged number (`events.jsonl`), the last value of each
number (`summary.json`) and the trained student (`student.pt`). A completed run is
**skipped** when its cell runs again, so you can stop at any point and rerun the cells
to carry on.

## Setup

`lwfm` and `experiments` aren't installed packages, so the repository root goes on the
import path. Open this notebook from `experiments/kd/`.

All the choices for the notebook are in the second cell:

- **How much to run**: three modes, from cheapest to full.
  - `SMOKE = True`: checks the wiring of every cell in a few minutes. Random weights,
    two training steps per run, one validation batch of 8 images; writes to
    `runs/smoke/`. Its numbers mean nothing.
  - `LOCAL_TRIAL` (on by default without a CUDA GPU): the real pipeline, with real
    weights and data, but with short budgets. 3 / 6 / 10 epochs instead of 25 / 100 / 300, each epoch
    capped at 30 steps of 64 images, evaluation on 1,024 random images, and 1 seed.
    It writes to `runs/local/`. Roughly a day on a laptop CPU. Its numbers are a first
    look, not results.
  - both `False` (the default on a CUDA GPU, e.g. the cluster): the full experiments,
    25 / 100 / 300 epochs and 3 seeds, written to `runs/`. On a CPU they would take
    months.
- **Device**: `pick_device()` picks CUDA, then MPS (Mac GPU), then the CPU. A file
  `.kd-device` at the repository root holding e.g. `cpu` overrides it. It is listed in
  `.git/info/exclude`, so it is never committed.
- **`SPLIT`**: which classes are used.
  - **10 + 40** (default): the 10 named classes plus 40 random "other" classes. Runs
    are ~4× faster, and the teacher's "other" signal is cleaner (see E0a).
  - **10 + 190**: the notebook split, with all 190 other classes in "other". It is
    closer to the class imbalance expected in whole-slide images, so use it to
    confirm the final results.
- **`DATASET`**: to move to ImageNet-1k, do these once:
  ```
  python download/download_data.py imagenet-1k data/imagenet-1k   # ~140 GB
  ```
  ```python
  SPLIT = make_split("imagenet-1k", train_fraction=0.96, exist_ok=True).name
  ```
  then set `DATASET = "imagenet-1k"`. The rest follows from its entry in
  `parts.DATASETS`:
  - 224 px images with ImageNet augmentation
  - the standard ResNet-18 stem and ViT patch 16
  - no "other" class, so the teacher's 1000 logits are used unchanged
  - class weights of 1, top-5 added to the metrics, weight decay 1e-4
- **`STUDENTS_TO_RUN`**: every experiment runs once per student. A new student is one
  entry in `parts.STUDENTS`.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]  # this notebook lives in experiments/kd/
sys.path.insert(0, str(REPO))

In [ ]:
import json
from dataclasses import replace
from functools import partial

import matplotlib.pyplot as plt
import pandas as pd
import torch

from experiments.kd.parts import (
    DIST,
    DKD,
    FAST,
    KD,
    LOCAL,
    LOCAL_FAST,
    LOCAL_LONG,
    LOCAL_MAIN,
    LONG,
    MAIN,
    RUNS,
    STUDENTS,
    TEACHERS,
    EmbedCos,
    EmbedMSE,
    Experiment,
    Setup,
    SplitData,
    ToStudentClasses,
    evaluate_teacher,
    fit_teacher_head,
    make_split,
    pick_device,
    results,
    run,
    student_node,
    teacher_embed_stats,
    tune_other_shift,
)
from experiments.kd.parts import (
    SMOKE as SMOKE_SETTINGS,
)
from lwfm.core.graph import ExecutionGraph
from lwfm.core.losses import LossComposer

device = pick_device()
SMOKE = False  # True: a few-minute check of the wiring (numbers mean nothing)
LOCAL_TRIAL = device.type != "cuda"  # short real run off-GPU; full runs on CUDA
DATASET = "tiny-imagenet"
NAMED = [f"{i:03d}" for i in range(10)]  # Tiny ImageNet folders 000-009
SPLIT = make_split(
    DATASET, keep=NAMED, num_other=40, train_fraction=0.7, exist_ok=True
).name  # 10 + 40; written once, then reused
# SPLIT = "tiny-70-30_5504132"  # 10 + 190, from examples/imagenet-splits.ipynb
STUDENTS_TO_RUN = ["resnet18", "vit_ti"]  # a CNN and a ViT
TEACHER = "resnet50"

data = SplitData(DATASET, SPLIT, workers=0 if SMOKE else 8)
mode = "smoke" if SMOKE else "local" if LOCAL_TRIAL else None
runs = (RUNS / mode if mode else RUNS) / data.name  # modes never mix their runs
SEEDS = (0, 1, 2) if mode is None else (0,)
if mode == "local":  # same budget shapes, far fewer epochs
    FAST, MAIN, LONG = LOCAL_FAST, LOCAL_MAIN, LOCAL_LONG

# Budgets: FAST (25 epochs, one-cycle) ranks settings in a sweep; MAIN (100 epochs,
# warmup + cosine) gives the reported results; LONG (300 epochs) checks whether KD's
# gain grows with training time. Each student adds its own optimiser settings, e.g.
# vit_ti trains with AdamW. Each run's manifest.json records the final settings.

# Passed to every run: device, pretrained teachers?, smoke overrides, results folder
overrides = {"smoke": SMOKE_SETTINGS, "local": LOCAL}.get(mode)
kw = dict(device=device, pretrained=not SMOKE, overrides=overrides, runs=runs)
eval_cfg = replace(MAIN, **(kw["overrides"] or {}))  # eval batch settings


def show(pattern: str, sort: str = "eval/val/bal_acc") -> pd.DataFrame:
    # One row per completed run matching runs/<pattern>, best first
    columns = [
        "run",
        "student",
        "teacher_head",
        "other_shift",
        "eval/val/bal_acc",
        "eval/val/macro_f1",
        "eval/test/bal_acc",
        "eval/test/macro_f1",
        "eval/test/top1",
        "eval/test/top5",
        "train/epoch_seconds",
        "cost/params_m",
        "cost/gflops",
        "cost/latency_ms_median",
    ]
    df = pd.DataFrame(results(pattern, runs))
    return df[[c for c in columns if c in df]].sort_values(sort, ascending=False)


def logged(run_name: str, name: str) -> list:
    # Every value logged under `name` in one run, in order (e.g. a per-epoch curve)
    lines = (runs / run_name / "events.jsonl").read_text().splitlines()
    return [e["value"] for e in map(json.loads, lines) if e["name"] == name]


print(f"device: {device}, mode: {mode or 'full'}, results in {runs}")

device: cpu, mode: local, results in /Users/ashleywebb/Uni/yr4/imagenet-classification/runs/local/tiny-imagenet/tiny-imagenet-10+40other-70-30_5504132


## The data

Labels 0–9 are the 10 named classes; the last label is "other". "Other" makes up 80 %
of the 10 + 40 split and 95 % of the 10 + 190 split. Two things follow:

- **Training** uses class-weighted cross-entropy, $w_c = N / (C \cdot n_c)$, so a
  mistake on a rare class costs more. On a balanced dataset every weight is 1.
- **Selection and reporting** use **balanced accuracy**, the mean of the per-class
  recalls. Always predicting "other" scores 80–95 % plain accuracy, but only
  1/11 = 9.1 % balanced accuracy, the same as chance.

`val` is a hold-out from the train folder and is used for **every** choice. `test` is
the official validation set and is used only for the final numbers.

In [ ]:
counts = {
    split: pd.Series([int(r["label"]) for r in data.rows(split)])
    .value_counts()
    .sort_index()
    for split in ("train", "val", "test")
}
print(
    f"{data.num_classes} classes, {data.spec.image_size} px, "
    f"{len(data.other_folders())} source classes in 'other'"
)
pd.DataFrame(
    {
        "folder": data.classes,
        "name": data.class_names(),
        **counts,
        "CE weight": data.class_weights().numpy().round(3),
    }
).head(20)

11 classes, 64 px, 40 source classes in 'other'


,folder,name,train,val,test,CE weight
0,000,goldfish,350,150,50,4.545
1,001,European fire salamander,350,150,50,4.545
2,002,bullfrog,350,150,50,4.545
3,003,tailed frog,350,150,50,4.545
4,004,American alligator,350,150,50,4.545
5,005,boa constrictor,350,150,50,4.545
6,006,trilobite,350,150,50,4.545
7,007,scorpion,350,150,50,4.545
8,008,black widow,350,150,50,4.545
9,009,tarantula,350,150,50,4.545


## How a run is put together

A `Setup` names the parts of one run, and `setup.build(data)` turns them into lwfm
graph nodes and loss bindings:

- **student**: a model from `parts.STUDENTS`, built for the dataset's image size. It
  outputs `student.logits` and `student.embed` (the pooled feature before the
  classifier).
- **teacher** (only when a KD loss needs it): frozen. It reads the **same augmented
  image** as the student, resized to 224 px, so both see the same crop and flip. Its
  1000 ImageNet logits are mapped to the student's classes:
  - the named classes keep their own logits;
  - the ImageNet classes behind "other" are merged into one logit, minus a prior
    correction `other_shift` (chosen in E0a);
  - every other ImageNet class is dropped.
- **projector** (only for an embedding loss): `Linear(student dim → teacher dim) +
  LayerNorm`. It is used only in training, so it is never saved or counted in the
  student's cost.
- **losses**: weighted CE, plus the chosen KD loss. Each loss class lists the values it
  reads, e.g. `KD.reads == ["student.logits", "teacher.logits"]`.

Swapping the dataset, teacher, student or loss is a change to one argument. Here is an
E1 setup for each student, built with random weights only to show the wiring:

In [ ]:
batch = next(iter(data.loader("val", 4)))
for student in STUDENTS_TO_RUN:
    setup = Setup(
        student, TEACHER, logit=partial(KD, T=4), alpha=0.9
    )  # 0.1·CE + 0.9·KD
    nodes, losses = setup.build(data, pretrained=False)
    graph = ExecutionGraph(nodes)
    print(student, graph.preflight(batch))
for b in losses:
    print(f"loss {b.name}: weight {b.weight}, reads {b.inputs}")
print("students:", list(STUDENTS), "| teachers:", list(TEACHERS))

resnet18 {'student.logits': (4, 11), 'student.embed': (4, 512), 'teacher.logits': (4, 11), 'teacher.embed': (4, 2048)}
vit_ti {'student.logits': (4, 11), 'student.embed': (4, 192), 'teacher.logits': (4, 11), 'teacher.embed': (4, 2048)}
loss ce: weight 0.1, reads ['student.logits', 'targets.class']
loss KD: weight 0.9, reads ['student.logits', 'teacher.logits']
students: ['resnet18', 'vit_ti'] | teachers: ['resnet50', 'resnet34', 'vit_b16', 'vit_b16_swag', 'dinov2_b14']


### Sanity checks

These run in seconds and must pass before training anything:

1. With no shift, mapping the teacher's logits gives the same probabilities as a
   softmax over just the split's classes, with "other" as the sum of its classes.
2. Identical student and teacher logits give KD, DKD and DIST ≈ 0. DIST is ~1e-3
   rather than 0, because of a small ε that keeps it numerically stable.
3. Class-weighted KD with all weights equal is plain KD.
4. After `backward()`, the frozen teacher has no gradients, while the student does.

In [ ]:
# 1. Teacher logits -> student classes
keep, other = data.teacher_classes(), data.teacher_other_classes()
z = torch.randn(4, 1000)
p_split = z[:, keep + other].softmax(
    1
)  # the teacher, restricted to the split's classes
p = ToStudentClasses(keep, other or None, shift=0.0)({"logits": z})["logits"].softmax(1)
assert torch.allclose(p[:, : len(keep)], p_split[:, : len(keep)], atol=1e-6)
if other:
    assert torch.allclose(p[:, -1], p_split[:, len(keep) :].sum(1), atol=1e-6)

# 2. Identical logits -> ~0 loss
z, y = torch.randn(8, data.num_classes), torch.randint(0, data.num_classes, (8,))
print(f"KD {KD()(z, z):.2e}, DKD {DKD()(z, z, y):.2e}, DIST {DIST()(z, z):.2e}")

# 3. Weighted KD with equal weights == plain KD
z2 = torch.randn(8, data.num_classes)
equal = torch.ones(data.num_classes)
assert torch.allclose(KD(weight=equal)(z, z2, y), KD()(z, z2))

# 4. Gradients reach the student only (graph and losses from the cell above)
total, terms = LossComposer(losses)(batch, graph(batch))
total.backward()
assert all(p.grad is None for p in graph.nodes["teacher"].parameters())
assert all(p.grad is not None for p in graph.nodes["student"].parameters())
print(
    "all checks passed; loss terms:", {k: round(v.item(), 3) for k, v in terms.items()}
)

KD 5.36e-07, DKD 1.49e-08, DIST 5.96e-07
all checks passed; loss terms: {'ce': 2.398, 'KD': 31.01}


## E0a: The teachers alone (inference only, no training)

Each teacher is scored exactly as it is used in KD: on the student's images resized to
224 px (upsampled from 64 px on Tiny ImageNet), with its logits mapped to the student's
classes. **Nothing is trained here.**

**Why the "other" logit needs correcting.** A softmax treats all its classes as equally
likely in advance. Merging $n$ classes into "other" therefore makes "other" $n$ times
likelier than a named class before the image is even seen. On blurry upsampled 64 px
images the teacher is unsure, so this prior wins, and the teacher calls most
named-class images "other". Measured on val with ResNet-50:

| Split | Teacher, no correction (bal. acc / macro-F1) | Teacher, tuned correction |
|---|---|---|
| 10 + 40 | 0.51 / 0.58 | 0.69 / 0.69 |
| 10 + 190 | 0.37 / 0.44 | 0.61 / 0.61 |

The fix subtracts a constant from the merged logit: $z_\text{other} =
\operatorname{logsumexp}(z_\text{other classes}) - b$.

- $b = \log n$ gives "other" the same prior as each named class. That maximises
  balanced accuracy, but it pulls "other" images towards the named classes.
- So $b$ is **chosen on val by macro-F1**, which also counts that loss of precision,
  once per teacher, from a single pass of the teacher.

Fewer "other" classes also means less to correct, which is one reason the 10 + 40 split
gives the cleaner teacher.

All teachers come from torchvision, except DINOv2, which downloads its code and weights
from GitHub through `torch.hub`. The teachers' accuracy here is optimistic, because
Tiny ImageNet's images come from ImageNet's training set.

In [ ]:
shifts = {}  # teacher -> chosen other_shift (None when there is no "other" class)
for teacher in TEACHERS:
    if SMOKE and teacher == "dinov2_b14":
        continue  # needs network access even with random weights
    if data.has_other:
        rows = tune_other_shift(
            teacher,
            data,
            device,
            pretrained=not SMOKE,
            eval_batches=eval_cfg.eval_batches,
            batch_size=eval_cfg.eval_batch_size,
        )
        shifts[teacher] = max(rows, key=lambda r: r["macro_f1"])["shift"]
        if teacher == TEACHER:
            display(pd.DataFrame(rows).round(3))  # the trade-off for the KD teacher
    else:
        shifts[teacher] = None
    evaluate_teacher(
        f"teachers/{teacher}/logits", teacher, data, other_shift=shifts[teacher], **kw
    )
if data.has_other:  # for comparison in E0c: the KD teacher with no correction at all
    evaluate_teacher(
        f"teachers/{TEACHER}/logits-no-shift", TEACHER, data, other_shift=0.0, **kw
    )
print("chosen shifts:", shifts)
show("teachers/*/logits")

,shift,bal_acc,macro_f1,p_other_on_other
0,0.000,0.524,0.589,0.925
1,0.461,0.594,0.633,0.892
2,0.922,0.632,0.661,0.847
3,1.383,0.668,0.681,0.790
4,1.844,0.711,0.709,0.721
5,2.306,0.737,0.720,0.643
6,2.767,0.748,0.710,0.560
7,3.228,0.748,0.676,0.475
8,3.689,0.746,0.593,0.393
9,4.150,0.737,0.501,0.315


eval/val/bal_acc = 0.7374
eval/val/macro_f1 = 0.7204
eval/val/top1 = 0.917
eval/test/bal_acc = 0.7004
eval/test/macro_f1 = 0.6769
eval/test/top1 = 0.9043
status = interrupted


KeyboardInterrupt: 

In [ ]:
# Test recall per class (rows) for each teacher (columns)
recall = {
    d.name: logged(f"teachers/{d.name}/logits", "test/per_class_recall")[0]
    for d in sorted((runs / "teachers").iterdir())
}
# Recalls are stored by folder id ("000"); show them by class name ("goldfish")
names = dict(zip(data.classes, data.class_names(), strict=True))
pd.DataFrame(recall).rename(index=names).round(3).head(20)

## E0b: The baseline students (trained, no teacher)

This is the only part of E0 that trains. Every later experiment asks "does adding a
teacher beat this?", so each student is trained here with **weighted CE only**, using
the same data, augmentation and budget as its KD runs.

- **ResNet-18** (11.2 M parameters): its convolutions build in locality, which suits
  small datasets.
- **ViT-Tiny** (5.5 M parameters, DeiT-Tiny shape): it lacks that built-in bias, so
  from scratch on this little data it is expected to trail ResNet-18. It trains with
  AdamW, warmup and gradient clipping, because ViTs train poorly with SGD. DeiT showed
  that ViTs gain the most from a CNN teacher, so how far KD closes the gap is itself a
  result.

Each student runs at every budget the later experiments use, so every comparison is like
for like: `FAST` ×1 (the reference for the sweeps), `MAIN` ×3 seeds and `LONG` ×1.

In [ ]:
for student in STUDENTS_TO_RUN:
    baseline = Setup(student)  # weighted CE only: no teacher, no KD loss
    run(f"no-teacher/e0/{student}/sweep/baseline_s0", baseline, data, FAST, **kw)
    for seed in SEEDS:
        run(
            f"no-teacher/e0/{student}/main/baseline_s{seed}",
            baseline,
            data,
            MAIN,
            seed=seed,
            **kw,
        )
    run(f"no-teacher/e0/{student}/long/baseline_s0", baseline, data, LONG, **kw)
show("no-teacher/e0/*/*/*")

## E0c: Heads trained on the frozen teacher

Instead of mapping the teacher's ImageNet logits to our classes (E0a), train a small
**head** on the frozen teacher's embedding, with our labels. The backbone is unchanged;
only the head is new.

- It learns "other" as an ordinary class, so no merge and no shift are needed.
- It learns what upsampled 64 px images look like in the teacher's feature space,
  where the ImageNet head was trained on sharp 224 px photos.
- Pathology foundation models have no class head, so they need exactly this step.

Two kinds of head, both trained on embeddings cached from one pass of the teacher over
the un-augmented train and val images:

| Head | Why try it |
|---|---|
| `linear` | The features were trained for a linear head; few parameters, hard to overfit |
| `mlp` (one hidden layer of 512, dropout) | Can undo some of the distortion from upsampling, but overfits sooner |

Each is trained with class-weighted CE for a few weight decays, with early stopping, all
chosen by **val macro-F1**. The table also shows each head's **confidence** (mean top
probability) on train and val. A head that is near-certain on train images has memorised
them, so its soft targets add little beyond the labels, which weakens KD.

On ImageNet-1k this section is skipped: the teacher's own 1000-way head already matches
the classes.

In [ ]:
HEAD_KINDS = ["linear", "mlp"] if data.has_other else []
head_rows = []
for kind in HEAD_KINDS:
    # Heads always use every train/val embedding (one teacher pass, then cached),
    # except in smoke mode, whose random-weight cache is kept separately
    head_rows += fit_teacher_head(
        TEACHER,
        data,
        device,
        kind,
        pretrained=not SMOKE,
        max_batches=eval_cfg.eval_batches if SMOKE else None,
        batch_size=eval_cfg.eval_batch_size,
    )
    evaluate_teacher(
        f"teachers/{TEACHER}/{kind}", TEACHER, data, teacher_head=kind, **kw
    )
pd.DataFrame(head_rows).round(3)  # one row per head and weight decay

### Comparing the teacher variants

**At teacher level:** each variant of the KD teacher, scored alone on val and test:

- `logits-no-shift`: the mapped ImageNet logits, uncorrected
- `logits`: the mapped ImageNet logits, with the shift from E0a
- `linear`, `mlp`: the trained heads

In [ ]:
show(f"teachers/{TEACHER}/*")

**At student level, which is what matters:** a better teacher on its own doesn't
guarantee a better student. Each variant teaches each student with the same standard KD
setting ($T = 4$, $\alpha = 0.9$) on the `FAST` budget. Compare with the baselines'
`FAST` runs (`no-teacher/e0/<student>/sweep`). The variant whose students score best on
val becomes the teacher for E1–E3.

In [ ]:
variants = {"logits": dict(other_shift=shifts[TEACHER])}  # mapped ImageNet logits
variants |= {kind: dict(teacher_head=kind) for kind in HEAD_KINDS}

for variant, teacher_args in variants.items():
    for student in STUDENTS_TO_RUN:
        setup = Setup(
            student, TEACHER, logit=partial(KD, T=4), alpha=0.9, **teacher_args
        )
        run(
            f"{TEACHER}-{variant}/e0c/{student}/sweep/kd_T4_a0.9_s0",
            setup,
            data,
            FAST,
            **kw,
        )

check = pd.DataFrame(
    results("*/e0c/*/sweep/*", runs) + results("no-teacher/e0/*/sweep/*", runs)
)
check["teacher"] = check["run"].str.split("/").str[0]
check.pivot_table(index="teacher", columns="student", values="eval/val/bal_acc").round(
    4
)

In [ ]:
# The variant with the best mean student val balanced accuracy teaches E1-E3
scores = (
    check[check["teacher"] != "no-teacher"]
    .groupby("teacher")["eval/val/bal_acc"]
    .mean()
)
TAG = scores.idxmax()  # e.g. "resnet50-linear"
VARIANT = TAG.removeprefix(f"{TEACHER}-")
S = partial(
    Setup, teacher=TEACHER, **variants[VARIANT]
)  # Setup with the teacher filled in
print(f"E1-E3 use {TAG}; runs go to runs/.../{TAG}/")

## E1: Logit KD

Loss: $(1-\alpha)\,\mathrm{CE} + \alpha\, T^2\, \mathrm{KL}\big(\mathrm{softmax}(z_t/T)\,\|\,\mathrm{softmax}(z_s/T)\big)$.

- A higher **T** softens both distributions, so the student learns how the teacher
  ranks the wrong classes, not only its top guess.
- **α** sets how much the student follows the teacher rather than the labels.
- **Class weighting** (`wkd_...`): every setting also runs with the KD term weighted
  by class, like the CE: each image's KL is multiplied by the weight of its true
  class, $\sum_i w_{y_i} \mathrm{KL}_i / \sum_i w_{y_i}$. Plain KD is dominated by
  "other", which is 80 % of the images and of the teacher's probability mass. In the
  debug run, KD students at α = 0.9 first learned to always predict "other", while
  their weighted CE rose. Weighting gives each class an equal share of the KD loss.
  The table after the sweep pairs each setting with its weighted version.

An `Experiment` is a dict of named setups:

- `sweep` trains each one on the `FAST` budget, with 1 seed.
- `confirm` takes the best on **val** balanced accuracy and trains it on `MAIN` with
  `SEEDS` (3 seeds, or 1 in a local trial), plus once on `LONG`.

Each student gets its own experiment, and its own best setting. Weighting is only
implemented for KD, so if a weighted setting wins E1, E2 compares it with unweighted
DKD and DIST: keep that in mind when reading E2.

In [ ]:
e1 = {
    student: Experiment(
        f"{TAG}/e1/{student}",
        {
            f"{'w' if weighted else ''}kd_T{T}_a{alpha}": S(
                student,
                logit=partial(KD, T=T),
                alpha=alpha,
                weighted_kd=weighted,  # wkd_...: KD weighted by class, like the CE
            )
            for T in (1, 2, 4)
            for alpha in (0.5, 0.9)
            for weighted in (False, True)
        },
        long_check=True,
        seeds=SEEDS,
        run_kwargs=kw,
    )
    for student in STUDENTS_TO_RUN
}
for exp in e1.values():
    exp.sweep(data, FAST)
show(f"{TAG}/e1/*/sweep/*")  # compare with no-teacher/e0/<student>/sweep

**Does class weighting help?** Each row is one $(T, \alpha)$, with the plain and the
weighted KD side by side for each student (`FAST` budget). Weighting helps where the
weighted column has the higher val balanced accuracy. Expect the gap to be largest at
α = 0.9, where the KD term has the most weight. The macro-F1 columns check that
weighting does not just trade the "other" class away.

In [ ]:
sweep = pd.DataFrame(results(f"{TAG}/e1/*/sweep/*", runs))
setting = sweep["run"].str.split("/").str[-1].str.replace(r"_s\d+$", "", regex=True)
sweep["KD"] = setting.str.startswith("w").map({False: "plain", True: "weighted"})
sweep["T, α"] = setting.str.removeprefix("w").str.removeprefix("kd_")
sweep.pivot_table(
    index="T, α",
    columns=["student", "KD"],
    values=["eval/val/bal_acc", "eval/val/macro_f1"],
).round(4)

In [ ]:
e1_best = {student: exp.confirm(data, MAIN, LONG)[0] for student, exp in e1.items()}
print("best E1 setting per student:", e1_best)
show(f"{TAG}/e1/*/main/*")

## E2: Swap the logit loss

Same graph, and the same weighted CE at weight 1. Only the KD term changes:

- **DKD** splits KD into a target-class part (TCKD, weight α) and a non-target part
  (NCKD, weight β). Its weight ramps from 0 to 1 over the first 20 epochs. Two settings:
  β = 0.5 and β = 8.
- **DIST** matches correlations rather than exact probabilities: across classes for
  each sample, and across the batch for each class.
- **Each student's best KD setting from E1**, as the reference.

With only 11 classes, NCKD and DIST's per-class term have little to work with, so gains
over KD may be small; they matter more on ImageNet-1k. The top **two** go on to 3 seeds.

In [ ]:
e2 = {
    student: Experiment(
        f"{TAG}/e2/{student}",
        {
            e1_best[student]: e1[student].setups[e1_best[student]],
            "dkd_b0.5": S(student, logit=partial(DKD, alpha=0.5, beta=0.5)),
            "dkd_b8": S(student, logit=partial(DKD, alpha=0.5, beta=8.0)),
            "dist": S(student, logit=partial(DIST, beta=1.0, gamma=1.0)),
        },
        top_k=2,
        seeds=SEEDS,
        run_kwargs=kw,
    )
    for student in STUDENTS_TO_RUN
}
for exp in e2.values():
    exp.sweep(data, FAST)
show(f"{TAG}/e2/*/sweep/*")

In [ ]:
e2_best = {student: exp.confirm(data, MAIN, LONG) for student, exp in e2.items()}
print("best two E2 losses per student:", e2_best)
show(f"{TAG}/e2/*/main/*")

## E3: Embedding KD

The student's pooled feature goes through the projector into the teacher's 2048-d
space, and is pulled towards the teacher's pooled feature. Embeddings don't involve
the class mapping, so the "other" correction plays no part here.

| Setup | Loss |
|---|---|
| `ce_cos` | CE + $1 - \cos(P(h_s), h_t)$: matches direction only |
| `ce_mse` | CE + MSE to the teacher feature, standardised per dimension with train-set statistics |
| `<best E2>+cos` | the student's best logit loss from E2, plus the cosine term |
| `cos_only` | the cosine term **only, with no labels**. A linear probe is then trained on the frozen student embedding, and scored. This is the label-free setting pathology will need |

`ce_mse` needs the teacher feature's per-dimension mean and std. They are computed once
in a pass over the un-augmented train set, then cached in `runs/cache/`. Every 10 epochs,
each run also logs `eval/val/embed_cos`, the val cosine between $P(h_s)$ and $h_t$. It
should rise during training.

In [ ]:
stats = teacher_embed_stats(
    TEACHER, data, device, pretrained=not SMOKE, max_batches=1 if SMOKE else None
)
e3 = {}
for student in STUDENTS_TO_RUN:
    logit_best = e2_best[student][0]
    e3[student] = Experiment(
        f"{TAG}/e3/{student}",
        {
            "ce_cos": S(student, embed=EmbedCos),
            "ce_mse": S(student, embed=partial(EmbedMSE, *stats)),
            f"{logit_best}+cos": replace(
                e2[student].setups[logit_best], embed=EmbedCos
            ),
            "cos_only": S(
                student, embed=EmbedCos, ce=False
            ),  # label-free, then a probe
        },
        seeds=SEEDS,
        run_kwargs=kw,
    )
    e3[student].sweep(data, FAST)
show(f"{TAG}/e3/*/sweep/*")

In [ ]:
e3_best = {student: exp.confirm(data, MAIN, LONG)[0] for student, exp in e3.items()}
print("best E3 setup per student:", e3_best)
show(f"{TAG}/e3/*/main/*")

In [ ]:
# E3 checks: the embedding cosine rises, and the saved student needs no projector
for student in STUDENTS_TO_RUN:
    for d in sorted((runs / f"{TAG}/e3/{student}/sweep").iterdir()):
        cos = logged(f"{TAG}/e3/{student}/sweep/{d.name}", "eval/val/embed_cos")
        print(f"{student} {d.name}: embed_cos {[round(v, 3) for v in cos]}")
    model = student_node(student, data).module
    saved = next((runs / f"{TAG}/e3/{student}/main").iterdir()) / "student.pt"
    model.load_state_dict(torch.load(saved, weights_only=True))  # strict
    print(f"{student}: saved student loads without the projector")

## Results

One row per configuration: the mean, std and count over seeds. Read the **main** stage
for the reported numbers, and compare every row with that student's `no-teacher`
baseline.
Within a student, the cost is the same in every row, because KD changes only how the
student is trained, never the model.

In [ ]:
df = pd.DataFrame(results(runs=runs))
df = df[~df["run"].str.startswith("teachers/")]  # training runs only
parts = df["run"].str.split(
    "/"
)  # <teacher variant>/<experiment>/<student>/<stage>/<setup>_s<seed>
df["teacher"], df["experiment"], df["stage"] = parts.str[0], parts.str[1], parts.str[3]
df["config"] = parts.str[4].str.replace(r"_s\d+$", "", regex=True)
metrics = [
    "eval/test/bal_acc",
    "eval/test/macro_f1",
    "eval/test/top5",
    "eval/val/bal_acc",
    "train/epoch_seconds",
    "cost/params_m",
    "cost/gflops",
]
summary = df.groupby(["stage", "student", "teacher", "experiment", "config"])[
    [m for m in metrics if m in df]
].agg(["mean", "std", "count"])
summary.to_csv(runs / "summary.csv")
summary.round(4)

In [ ]:
# Val balanced accuracy per epoch on the MAIN budget, averaged over seeds
fig, axes = plt.subplots(
    1, len(STUDENTS_TO_RUN), figsize=(6 * len(STUDENTS_TO_RUN), 4), squeeze=False
)
main_runs = df[df["stage"] == "main"]
for ax, student in zip(axes[0], STUDENTS_TO_RUN, strict=True):
    for config, group in main_runs[main_runs["student"] == student].groupby(
        ["teacher", "experiment", "config"]
    ):
        curves = [logged(r, "eval/val/bal_acc") for r in group["run"]]
        n = min(map(len, curves))
        ax.plot(
            [sum(c[i] for c in curves) / len(curves) for i in range(n)],
            label="/".join(config),
        )
    ax.set(title=student, xlabel="epoch", ylabel="val balanced accuracy")
    ax.legend(fontsize=7)
plt.show()

For each experiment, write one paragraph describing any deviation from
`kd_plan_E0-E3.md`, and why.